# Submissao 1 - CNN1D
Notebook limpo para inferencia final: le CSV, preprocessa, carrega pesos e guarda submissao.

In [1]:
import sys
import os
from pathlib import Path

sys.path.append(os.path.abspath('..'))

import torch
import pandas as pd
import numpy as np
from src.data_processing import clean_text
from src.features import Vocabulary, texts_to_sequences
from src.models_pytorch.cnn1d import CNN1DClassifier

input_csv = Path('../submissions/submissao1/subm1.csv')
train_csv = Path('../data/processed/dataset_kaggle_daigt_processed.csv')
weights_path = Path('../saved_models/cnn1d_final.pth')
vocab_path = Path('../saved_models/vocab_cnn.pkl')
output_dir = Path('../submissions/submissao1/results')
output_csv = output_dir / 'submission_cnn1d.csv'

for required in [input_csv, train_csv, weights_path, vocab_path]:
    if not required.exists():
        raise FileNotFoundError(f'Ficheiro em falta: {required}')

output_dir.mkdir(parents=True, exist_ok=True)


df_train = pd.read_csv(train_csv, sep=';')
label_names = sorted(df_train['Label'].unique())
idx_to_label = {i: label for i, label in enumerate(label_names)}
num_classes = len(label_names)


df_sub = pd.read_csv(input_csv, sep=';')
df_sub['text_clean'] = df_sub['Text'].fillna('').apply(clean_text)

vocab = Vocabulary(max_words=1000)
vocab.load(str(vocab_path))

max_len = 32
X_sub_seq = texts_to_sequences(df_sub['text_clean'].tolist(), vocab, max_len=max_len)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = CNN1DClassifier(
    vocab_size=len(vocab),
    embedding_dim=100,
    n_filters=128,
    filter_sizes=[1, 2, 3],
    output_dim=5,
    dropout=0.45
)

model.load_state_dict(torch.load(weights_path, map_location=device))
model.to(device)
model.eval()

X_tensor = torch.tensor(X_sub_seq, dtype=torch.long)
batch_size = 64
pred_ids = []

with torch.no_grad():
    for i in range(0, len(X_tensor), batch_size):
        batch = X_tensor[i:i + batch_size].to(device)
        logits = model(batch)
        pred_ids.extend(logits.argmax(dim=1).cpu().numpy().tolist())

df_sub['Label'] = [idx_to_label[int(i)] for i in pred_ids]
df_out = df_sub[['ID', 'Label']]
df_out.to_csv(output_csv, sep=';', index=False)

print(f'Submissao CNN1D guardada em: {output_csv}')
print(df_out.head())

Submissao CNN1D guardada em: ../submissions/submissao1/results/submission_cnn1d.csv
     ID   Label
0  D2-1  OpenAI
1  D2-2  OpenAI
2  D2-3  OpenAI
3  D2-4  OpenAI
4  D2-5  OpenAI
